In [3]:
!nvidia-smi

Tue Nov 10 14:45:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
import torch
import platform

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Memória total:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )
else:
    print("Nenhuma GPU CUDA encontrada.")

Python: 3.13.16
PyTorch: 2.14.1+cu130
CUDA disponível: True
GPU: NVIDIA A100-SXM4-80GB
Memória total: 79.25 GB


In [3]:

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

# ==========================================
# CONFIGURAÇÕES
# ==========================================

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

# ==========================================
# VERIFICAR GPU
# ==========================================

assert torch.cuda.is_available(), "GPU CUDA não encontrada!"

print("GPU:", torch.cuda.get_device_name(0))
print("BF16 disponível:", torch.cuda.is_bf16_supported())

# ==========================================
# CARREGAR TOKENIZER
# ==========================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ==========================================
# CARREGAR MODELO EM BF16
# ==========================================

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="auto"
)

model.eval()

print("\nModelo carregado com sucesso!")
print("Tipo de precisão:", model.dtype)
print("Dispositivo:", model.device)


GPU: NVIDIA A100-SXM4-80GB
BF16 disponível: True


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Modelo carregado com sucesso!
Tipo de precisão: torch.bfloat16
Dispositivo: cuda:0


In [4]:

messages = [
    {
        "role": "user",
        "content": "Explique brevemente o que é um incêndio florestal."
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)


Um incêndio florestal é um grande foco de fogo que ocorre em áreas de vegetação rasteira, como florestas, campos de grama ou matas nativas. Esses incêndios podem ser causados por diversos fatores, incluindo ventos fortes, calor extremo, chuvas escassas ou até mesmo acidentes humanos.

Os incêndios florestais são perigosos e têm impactos significativos na biodiversidade, na saúde do solo e na vida humana. Eles podem levar à morte de animais e plantas, destruição de habitats, poluição do ar e da água, além de caus
